# Frente C: verificación de H1 a H4 (sección 3)

Cada comparación es paciente por paciente con la prueba de Wilcoxon de muestras apareadas. H1 y H2 usan funciones del Frente B y H4 del Frente A, pero el diseño experimental está en `evaluacion/hipotesis.py`. Figuras con prefijo `h_`.

In [ ]:
# Entorno común a todos los notebooks: se corre desde codigo/ para que DIR_DATOS
# ("data" o $ESTADIOS_DATOS) apunte al mismo lugar que los scripts.
import os
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

# Si el paquete no está instalado con `pip install -e .`, se usa directamente codigo/src.
# Las dependencias (numpy, scipy, pandas, matplotlib, mne, scikit-learn) sí tienen que
# estar instaladas en el Python de este kernel.
try:
    import estadios  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd() / "src"))
FIGURAS = Path.cwd().parent / "informe" / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt

from estadios.comun.config import Config
from estadios.comun.constantes import NOMBRES_ESTADIOS, N_ESTADIOS, SIN_ANOTAR
from estadios.datos.carga import ids_disponibles

USAR_DATOS_REALES = len(ids_disponibles()) > 0
print("Datos reales" if USAR_DATOS_REALES else "Sin datos descargados: se usan los sintéticos")

In [ ]:
def guardar(fig, nombre):
    """Guarda la figura en informe/figuras/ para incluirla con \\includegraphics."""
    fig.savefig(FIGURAS / f"{nombre}.pdf", bbox_inches="tight")

In [ ]:
from dataclasses import replace

from estadios.evaluacion import hipotesis

def mostrar(nombre, comparacion, etiqueta_a, etiqueta_b, ylabel):
    print(f"{nombre}: p = {comparacion.p_valor:.4f}")
    fig, ax = plt.subplots(figsize=(4, 4))
    for a, b in zip(comparacion.variante_a, comparacion.variante_b):
        ax.plot([0, 1], [a, b], c="gray", alpha=0.5)
    ax.boxplot([comparacion.variante_a, comparacion.variante_b], positions=[0, 1],
               tick_labels=[etiqueta_a, etiqueta_b])
    ax.set(ylabel=ylabel, title=f"{nombre} (p = {comparacion.p_valor:.3g})")
    guardar(fig, f"h_{nombre.lower()}")

## H1: el multitaper varía menos que Welch entre épocas de un mismo estadio

Con los registros sintéticos este resultado no sirve: su potencia por banda es exactamente la misma en todas las épocas de un estadio, lo que favorece a Welch. H1 se evalúa con datos reales.

In [ ]:
from estadios.comun.sintetico import registro_sintetico
from estadios.datos.carga import cargar_registro
from estadios.datos.particion import particion_por_paciente

if USAR_DATOS_REALES:
    _, ids_prueba = particion_por_paciente(ids_disponibles())
    registros = [cargar_registro(i) for i in ids_prueba]
else:
    registros = [registro_sintetico(semilla=s) for s in range(6)]

# Por banda (lo que usan los descriptores) y frecuencia por frecuencia (donde el
# multitaper debería mostrar su ventaja: integrar una banda ya promedia el ruido).
h1 = hipotesis.h1_variabilidad_espectral(registros)
mostrar("H1_bandas", h1, "Multitaper", "Welch", "Variación entre épocas (log10)")
h1_f = hipotesis.h1_variabilidad_espectral(registros, por_frecuencia=True)
mostrar("H1_frecuencias", h1_f, "Multitaper", "Welch", "Variación entre épocas (log10)")

## H2: Woody recupera la forma de la onda lenta mejor que el promedio ordinario

In [ ]:
from estadios.comun.constantes import EEG_F4, W

eeg_vigilia = []
for r in registros:
    n = int(30 * r.fs)
    epocas_w = np.flatnonzero(r.estadios == W)[:20]
    eeg_vigilia.append(np.concatenate([r.senales[EEG_F4][i * n:(i + 1) * n] for i in epocas_w]))

h2 = hipotesis.h2_recuperacion_de_forma(eeg_vigilia, registros[0].fs)
mostrar("H2", h2, "Woody", "Promedio", "Error de forma")

## H3: CSP mejora el F1 de N1 y REM

Con CSP está activo el desempate N1/REM de `reglas.desempatar_con_csp`.

In [ ]:
from estadios.comun.sintetico import tabla_sintetica
from estadios.pipeline import obtener_tabla

cfg = Config()
if USAR_DATOS_REALES:
    ids_entrenamiento, ids_prueba = particion_por_paciente(ids_disponibles())
    entrenamiento = [obtener_tabla(i, cfg) for i in ids_entrenamiento]
    prueba = [obtener_tabla(i, cfg) for i in ids_prueba]
else:
    entrenamiento = [tabla_sintetica(semilla=s) for s in range(8)]
    prueba = [tabla_sintetica(semilla=s) for s in range(100, 106)]

h3 = hipotesis.h3_aporte_csp(entrenamiento, prueba, cfg)
for estadio, comparacion in h3.items():
    mostrar(f"H3_{estadio}", comparacion, "Con CSP", "Sin CSP", f"F1 de {estadio}")

## H4: la remoción de artefactos y la HRV mejoran el acuerdo

Primero se verifica que el residuo cardíaco tienda a cero; después se compara κ entre dos configuraciones, cada una con su propio caché de descriptores.

In [ ]:
try:
    h4_residuo = hipotesis.h4_residuo_cardiaco(registros, cfg)
    mostrar("H4_residuo", h4_residuo, "Después", "Antes", "Promedio alrededor del latido (µV)")
except NotImplementedError as pendiente:
    print(f"Pendiente del Frente A: {pendiente}. Esta celda corre sola cuando esté implementado.")

In [ ]:
from estadios.comun.config import GRUPOS_DESCRIPTORES

cfg_sin = replace(
    cfg,
    preprocesamiento=replace(cfg.preprocesamiento, remover_ocular=False, remover_cardiaco=False),
    clasificacion=replace(cfg.clasificacion,
                          grupos=tuple(g for g in GRUPOS_DESCRIPTORES if g != "hrv")),
)
if USAR_DATOS_REALES:
    sin_etapas = ([obtener_tabla(i, cfg_sin) for i in ids_entrenamiento],
                  [obtener_tabla(i, cfg_sin) for i in ids_prueba])
else:
    sin_etapas = (entrenamiento, prueba)  # Con sintéticos solo se prueba que corra.

h4_kappa = hipotesis.h4_aporte_al_acuerdo((entrenamiento, prueba), sin_etapas, cfg)
mostrar("H4_kappa", h4_kappa, "Con etapas", "Sin etapas", "κ de Cohen")